# State of Data — EDA: Gênero (2021–2024)
Análise exploratória com gráficos interativos (Plotly).

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Constantes globais ────────────────────────────────────────────────────
CORES = {'Masculino': '#517493', 'Feminino': '#64313E'}
ANOS  = ['2021', '2022', '2023', '2024']
TEMPLATE = 'plotly_white'

FAIXA_SALARIAL_ORDEM = [
    'R$1k-2k','R$2k-3k','R$3k-4k','R$4k-6k','R$6k-8k',
    'R$8k-12k','R$12k-16k','R$16k-20k','R$20k-25k',
    'R$25k-30k','R$30k-40k','R$40k+',
]
NIVEL_ENSINO_ORDEM = [
    'Não tenho graduação formal','Estudante de Graduação',
    'Graduação/Bacharelado','Especialização Lato Sensu','Mestrado','Doutorado ou Phd',
]
NIVEL_SENIORIDADE_ORDEM = ['Júnior','Pleno','Sênior','Gestor']

# ── Carrega o DataFrame já tratado ────────────────────────────────────────
df = pd.read_excel('df_full.xlsx')
df['ano_pesquisa'] = df['ano_pesquisa'].astype(str)

# Restaura as categóricas ordenadas após leitura do Excel
df['faixa_salarial'] = pd.Categorical(df['faixa_salarial'], categories=FAIXA_SALARIAL_ORDEM, ordered=True)
df['nivel_ensino']   = pd.Categorical(df['nivel_ensino'],   categories=NIVEL_ENSINO_ORDEM,   ordered=True)
df['nivel']          = pd.Categorical(df['nivel'],          categories=NIVEL_SENIORIDADE_ORDEM, ordered=True)

print(f'Linhas: {len(df):,} | Anos: {sorted(df["ano_pesquisa"].unique())}')
df.head(2)

Linhas: 17,295 | Anos: ['2021', '2022', '2023', '2024']


,ano_pesquisa,idade,faixa_idade,genero,estado,uf,regiao,regiao_origem,mudou_estado,nivel_ensino,area_formacao,situacao,setor,cargo,gestor,nivel,faixa_salarial,tempo_area_dados,tempo_area_ti,modalidade
0,2021,38.0,35-39,Masculino,Ceará (CE),CE,Nordeste,NaN,0.0,Especialização Lato Sensu,Química / Física,Empregado (CLT),Marketing,NaN,1.0,Gestor,R$4k-6k,Mais de 10 anos,Não tive experiência na área de TI/Engenharia ...,Modelo 100% presencial
1,2021,39.0,35-39,Masculino,Bahia (BA),BA,Nordeste,Sudeste,1.0,Especialização Lato Sensu,Economia / Administração / Finanças / Negócios,Empreendedor ou Empregado (CNPJ),Consultoria,NaN,1.0,Gestor,R$6k-8k,de 2 a 3 anos,Não tive experiência na área de TI/Engenharia ...,Modelo híbrido flexível (o funcionário tem lib...


## Funções auxiliares

In [2]:
def prop_dentro_genero(df, col_grupo, col_genero='genero'):
    """
    Proporção de cada categoria de col_grupo DENTRO de cada gênero.
    Ou seja: de todas as mulheres, quantas % são Júnior? Pleno? etc.
    Retorna DataFrame com index=col_grupo, columns=gêneros.
    """
    ct = df.groupby([col_genero, col_grupo], observed=True).size().unstack(fill_value=0)
    return (ct.div(ct.sum(axis=1), axis=0) * 100).T  # transpõe para index=grupo


def prop_genero_dentro_grupo(df, col_grupo, col_genero='genero'):
    """
    Proporção de gênero DENTRO de cada categoria (soma = 100% por linha).
    Ou seja: dentro dos Engenheiros de Dados, quantas % são mulheres?
    Retorna DataFrame com index=col_grupo, columns=gêneros.
    """
    ct = df.groupby([col_grupo, col_genero], observed=True).size().unstack(fill_value=0)
    return ct.div(ct.sum(axis=1), axis=0) * 100


def legenda_unica(fig):
    """Remove entradas duplicadas de legenda (mantém só a primeira ocorrência de cada nome)."""
    vistos = set()
    for trace in fig.data:
        if trace.name in vistos:
            trace.showlegend = False
        else:
            vistos.add(trace.name)
            trace.showlegend = True
    return fig


def layout_padrao(fig, titulo, height=500, **kwargs):
    fig.update_layout(
        title=dict(text=titulo, font=dict(size=16)),
        template=TEMPLATE,
        height=height,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        **kwargs
    )
    return fig

---
## Gráfico 1 — Faixa Salarial × Gênero por Ano
> Barras horizontais agrupadas, um subplot por ano. Eixo Y compartilhado com a ordem correta das faixas.

In [3]:
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=ANOS,
    shared_yaxes=True
)

for i, ano in enumerate(ANOS, 1):
    df_ano = df[df['ano_pesquisa'] == ano]
    df_plot = (
        df_ano.groupby(['faixa_salarial', 'genero'], observed=True)
        .size().reset_index(name='n')
        .pivot(index='faixa_salarial', columns='genero', values='n').fillna(0)
        .reindex(FAIXA_SALARIAL_ORDEM)  # garante ordem correta
    )
    for genero in ['Feminino', 'Masculino']:
        vals = df_plot[genero] if genero in df_plot.columns else [0]*len(df_plot)
        fig.add_trace(go.Bar(
            y=df_plot.index.tolist(),
            x=vals,
            name=genero,
            orientation='h',
            marker_color=CORES[genero],
            text=[str(int(v)) for v in vals],
            textposition='auto',
        ), row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Distribuição de Faixa Salarial por Gênero (2021–2024)',
                    height=600, barmode='group')
fig.update_xaxes(title_text='Respondentes')
fig.update_yaxes(title_text='Faixa Salarial', col=1)
fig.show()

---
## Gráfico 2 — Pirâmide de Senioridade × Gênero por Ano
> Proporção calculada **dentro do gênero**: de todas as mulheres, quantas % estão em cada nível?  
> Feminino vai para esquerda (valores negativos), Masculino para direita.

In [4]:
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=ANOS,
    shared_yaxes=True
)

for i, ano in enumerate(ANOS, 1):
    df_ano = df[df['ano_pesquisa'] == ano].dropna(subset=['nivel'])
    prop = prop_dentro_genero(df_ano, 'nivel').reindex(NIVEL_SENIORIDADE_ORDEM)

    for genero in ['Feminino', 'Masculino']:
        vals = prop[genero] if genero in prop.columns else pd.Series([0]*len(prop))
        x_vals = -vals if genero == 'Feminino' else vals  # espelha feminino
        fig.add_trace(go.Bar(
            y=prop.index.tolist(),
            x=x_vals,
            name=genero,
            orientation='h',
            marker_color=CORES[genero],
            text=[f'{int(round(v))}%' for v in vals],
            textposition='outside',
        ), row=1, col=i)

    # Linha de referência em 0
    fig.add_shape(
        type='line', x0=0, x1=0,
        y0=-0.5, y1=len(NIVEL_SENIORIDADE_ORDEM)-0.5,
        line=dict(color='black', width=1, dash='dot'),
        row=1, col=i
    )
    fig.update_xaxes(ticksuffix='%', range=[-100, 100], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Proporção de Nível de Senioridade por Gênero (2021–2024) — dentro do gênero',
                    height=400, barmode='relative')
fig.show()

---
## Gráfico 3 — Escolaridade × Gênero por Ano (proporção empilhada)
> Proporção de gênero **dentro de cada nível de ensino**: dentro dos doutorados, quantas % são mulheres?

In [5]:
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=ANOS,
    shared_yaxes=True
)

for i, ano in enumerate(ANOS, 1):
    df_ano = df[df['ano_pesquisa'] == ano].dropna(subset=['nivel_ensino'])
    prop = prop_genero_dentro_grupo(df_ano, 'nivel_ensino').reindex(NIVEL_ENSINO_ORDEM)

    for genero in ['Feminino', 'Masculino']:
        vals = prop[genero] if genero in prop.columns else [0]*len(prop)
        fig.add_trace(go.Bar(
            y=prop.index.tolist(),
            x=vals,
            name=genero,
            orientation='h',
            marker_color=CORES[genero],
            text=[f'{v:.0f}%' for v in vals],
            textposition='inside',
        ), row=1, col=i)

    fig.update_xaxes(ticksuffix='%', range=[0, 100], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Escolaridade × Gênero — proporção de gênero dentro de cada nível (2021–2024)',
                    height=500, barmode='stack')
fig.update_yaxes(title_text='Nível de Ensino', col=1)
fig.show()

---
## Gráfico 4 — Área de Formação × Gênero (evolução por ano)
> Grade de subplots: uma célula por área, barras agrupadas por ano e gênero.

In [6]:
df_area = df.dropna(subset=['area_formacao'])
df_contagem = df_area.groupby(['area_formacao', 'ano_pesquisa', 'genero']).size().reset_index(name='n')

areas = df_contagem.groupby('area_formacao')['n'].sum().sort_values(ascending=False).index.tolist()
n_cols = 3
n_rows = -(-len(areas) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=areas,
    shared_xaxes=True,
    horizontal_spacing=0.07,
    vertical_spacing=0.12
)

for idx, area in enumerate(areas):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    for genero in ['Feminino', 'Masculino']:
        df_filtrado = df_contagem[(df_contagem['area_formacao'] == area) & (df_contagem['genero'] == genero)]
        y_vals = [df_filtrado[df_filtrado['ano_pesquisa'] == ano]['n'].values[0]
                  if not df_filtrado[df_filtrado['ano_pesquisa'] == ano].empty else 0
                  for ano in ANOS]
        fig.add_trace(go.Bar(
            x=ANOS, y=y_vals,
            name=genero,
            marker_color=CORES[genero],
            text=y_vals,
            textposition='auto',
        ), row=row, col=col)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Área de Formação × Gênero — evolução por ano',
                    height=min(1000, n_rows * 250), width=1200, barmode='group')
fig.show()

---
## Gráfico 5 — Escolaridade × Gênero (contagem absoluta por ano)
> Barras horizontais agrupadas com contagens absolutas, por ano.

In [7]:
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=ANOS,
    shared_yaxes=True
)

df_escol = df.dropna(subset=['nivel_ensino'])
df_contagem = df_escol.groupby(['ano_pesquisa', 'nivel_ensino', 'genero'], observed=True).size().reset_index(name='n')
max_val = df_contagem['n'].max()

for i, ano in enumerate(ANOS, 1):
    df_ano = df_contagem[df_contagem['ano_pesquisa'] == ano]
    for genero in ['Feminino', 'Masculino']:
        counts = []
        for nivel in NIVEL_ENSINO_ORDEM:
            row_data = df_ano[(df_ano['nivel_ensino'] == nivel) & (df_ano['genero'] == genero)]
            counts.append(int(row_data['n'].values[0]) if not row_data.empty else 0)
        fig.add_trace(go.Bar(
            y=NIVEL_ENSINO_ORDEM,
            x=counts,
            name=genero,
            orientation='h',
            marker_color=CORES[genero],
            text=[str(c) for c in counts],
            textposition='outside',
        ), row=1, col=i)
    fig.update_xaxes(range=[0, max_val * 1.25], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Escolaridade × Gênero — contagem absoluta por ano',
                    height=600, barmode='group')
fig.update_yaxes(title_text='Nível de Ensino', col=1)
fig.show()

---
## Gráfico 6 — Cargo × Gênero por Ano
> Um subplot por ano com contagens absolutas. Cargos ordenados por frequência total.

In [8]:
df_cargo = df[df['cargo'] != 'Outro'].dropna(subset=['cargo'])
df_contagem = df_cargo.groupby(['cargo', 'ano_pesquisa', 'genero']).size().reset_index(name='n')
cargos = df_contagem.groupby('cargo')['n'].sum().sort_values().index.tolist()  # menor → maior

fig = make_subplots(
    rows=len(ANOS), cols=1,
    subplot_titles=[f'Ano {a}' for a in ANOS],
    shared_xaxes=True,
    vertical_spacing=0.05
)

for i, ano in enumerate(ANOS, 1):
    df_ano = df_contagem[df_contagem['ano_pesquisa'] == ano]
    for genero in ['Feminino', 'Masculino']:
        df_g = df_ano[df_ano['genero'] == genero]
        x_vals = [df_g[df_g['cargo'] == c]['n'].values[0] if not df_g[df_g['cargo'] == c].empty else 0
                  for c in cargos]
        fig.add_trace(go.Bar(
            y=cargos, x=x_vals,
            name=genero,
            orientation='h',
            marker_color=CORES[genero],
            text=x_vals,
            textposition='auto',
        ), row=i, col=1)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Cargo × Gênero — evolução por ano',
                    height=max(900, len(ANOS) * 220), width=1000, barmode='group')
fig.update_xaxes(title_text='Quantidade', row=len(ANOS), col=1)
fig.show()

---
## Gráfico 7 — Modalidade de Trabalho × Gênero por Ano

In [9]:
df_modal = df.dropna(subset=['modalidade'])
df_contagem = df_modal.groupby(['modalidade', 'ano_pesquisa', 'genero']).size().reset_index(name='n')
modalidades = sorted(df_contagem['modalidade'].unique())

fig = make_subplots(
    rows=len(ANOS), cols=1,
    subplot_titles=[f'Ano {a}' for a in ANOS],
    shared_xaxes=True,
    vertical_spacing=0.05
)

for i, ano in enumerate(ANOS, 1):
    df_ano = df_contagem[df_contagem['ano_pesquisa'] == ano]
    for genero in ['Feminino', 'Masculino']:
        df_g = df_ano[df_ano['genero'] == genero]
        x_vals = [df_g[df_g['modalidade'] == m]['n'].values[0]
                  if not df_g[df_g['modalidade'] == m].empty else 0
                  for m in modalidades]
        fig.add_trace(go.Bar(
            y=modalidades, x=x_vals,
            name=genero,
            orientation='h',
            marker_color=CORES[genero],
            text=x_vals,
            textposition='auto',
        ), row=i, col=1)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Modalidade de Trabalho × Gênero — evolução por ano',
                    height=max(900, len(ANOS) * 220), width=1200, barmode='group')
fig.update_xaxes(title_text='Quantidade', row=len(ANOS), col=1)
fig.show()

---
## Gráfico 8 — Situação de Trabalho × Gênero (evolução por situação)
> Grade de subplots: uma célula por situação, barras agrupadas por ano e gênero.

In [10]:
df_situ = df.dropna(subset=['situacao'])
df_contagem = df_situ.groupby(['situacao', 'ano_pesquisa', 'genero']).size().reset_index(name='n')
situacoes = df_contagem.groupby('situacao')['n'].sum().sort_values(ascending=False).index.tolist()

n_cols = 3
n_rows = -(-len(situacoes) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=situacoes,
    shared_yaxes=False,
    horizontal_spacing=0.07,
    vertical_spacing=0.12
)

for idx, situacao in enumerate(situacoes):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    for genero in ['Feminino', 'Masculino']:
        df_filtrado = df_contagem[(df_contagem['situacao'] == situacao) & (df_contagem['genero'] == genero)]
        y_vals = [df_filtrado[df_filtrado['ano_pesquisa'] == ano]['n'].values[0]
                  if not df_filtrado[df_filtrado['ano_pesquisa'] == ano].empty else 0
                  for ano in ANOS]
        fig.add_trace(go.Bar(
            x=ANOS, y=y_vals,
            name=genero,
            marker_color=CORES[genero],
            text=y_vals,
            textposition='auto',
        ), row=row, col=col)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Situação de Trabalho × Gênero — evolução por ano',
                    height=min(1200, n_rows * 300), width=1300, barmode='group')
fig.show()

---
## Bônus — Evolução da representatividade feminina (2021–2024)

In [11]:
evolucao = (
    df.groupby(['ano_pesquisa', 'genero'])
    .size().unstack(fill_value=0)
    .assign(pct_feminino=lambda x: x['Feminino'] / (x['Feminino'] + x['Masculino']) * 100)
    .reindex(ANOS)
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ANOS,
    y=evolucao['pct_feminino'],
    mode='lines+markers+text',
    text=[f'{v:.1f}%' for v in evolucao['pct_feminino']],
    textposition='top center',
    line=dict(color=CORES['Feminino'], width=3),
    marker=dict(size=10),
    name='% Feminino'
))

fig = layout_padrao(fig, 'Evolução da Representatividade Feminina no Mercado de Dados (2021–2024)',
                    height=400)
fig.update_yaxes(ticksuffix='%', range=[0, 35], title_text='% Respondentes Feminino')
fig.update_xaxes(title_text='Ano da pesquisa')
fig.show()